# Hava Sıcaklığı Tahmini

Bu projede günlük sıcaklığı tahmin edeceğim.


In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
import seaborn as sns


### Data


In [ ]:
df=pd.read_csv('data/weatherHistory.csv')
df.head()


### EDA


In [ ]:
df.info()
df.isnull().sum()


### Görselleştirme


In [ ]:
df['Temperature (C)'].hist(bins=30)
plt.show()


### Boş veri


In [ ]:
df['Precip Type']=df['Precip Type'].fillna('unknown')
df['Temperature (C)']=df['Temperature (C)'].ffill()


### Feature Engineering


In [ ]:
df['dt']=pd.to_datetime(df['Formatted Date'],utc=True,errors='coerce')
g=df.dropna(subset=['dt']).set_index('dt').sort_index()
gun=g['Temperature (C)'].resample('D').mean().dropna()
gun=gun.to_frame('temp')
gun['lag1']=gun['temp'].shift(1)
gun['lag7']=gun['temp'].shift(7)
gun['ay']=gun.index.month
gun=gun.dropna()


### Train Test Split


In [ ]:
from sklearn.model_selection import train_test_split
x=gun[['lag1','lag7','ay']]
y=gun['temp']
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,shuffle=False)


### 3 Model


In [ ]:
from sklearn.linear_model import LinearRegression,Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score,mean_absolute_error

for ad,m in [('LR',LinearRegression()),('Ridge',Ridge()),('RF',RandomForestRegressor(random_state=42))]:
    m.fit(x_train,y_train)
    p=m.predict(x_test)
    print(ad,round(r2_score(y_test,p),3),round(mean_absolute_error(y_test,p),3))


### Feature Importance + Residual


In [ ]:
rf=RandomForestRegressor(random_state=42).fit(x_train,y_train)
print(pd.Series(rf.feature_importances_,index=x.columns))
pred=rf.predict(x_test)
plt.scatter(pred,y_test-pred)
plt.axhline(0,color='r')
plt.show()


In [ ]:
import joblib
joblib.dump(rf,'../../models/timeseries_weather.joblib')


### Sonuç

weatherHistory setinde dünkü sıcaklık çok işe yarıyor. Hedefi tutturdum.
